In [324]:
import torch
from torch import nn

d = nn.Dropout(0.5)
x = torch.ones(4, 6)
res = d(x)
res

tensor([[2., 2., 2., 2., 0., 2.],
        [2., 2., 2., 2., 0., 0.],
        [2., 0., 0., 0., 0., 0.],
        [2., 2., 0., 0., 2., 2.]])

Two things there:

*   Roughly half the entries are zeroed, independently per element — not a fixed count. Your last row happened to survive entirely.
*   Survivors are **2.0**, not 1.0. Scaled by `1/(1-p)` = 1/0.5.

That scaling keeps the expected value unchanged: each element is `2.0` half the time and `0` half the time, so it averages to the original `1.0`. Without it, every dropout layer would shrink activations and the network would see a systematically different scale.

In [325]:
torch.count_nonzero(res)

tensor(14)

In [326]:
d.eval()
print(d(x))

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])


Identity in eval — dropout is a pure passthrough at inference.

That's the payoff for the `1/(1-p)` scaling: because training already corrected the scale, inference needs no adjustment at all.

Concretely: this is exactly why `model.eval()` in `estimate_loss` stops being a no-op once you add dropout. Without it you'd be evaluating a randomly mutilated model and reporting the loss as real.

In [327]:
w = torch.softmax(torch.randn(4, 4), dim=-1)
print(w)
print(w.sum(dim=-1))
d.train()
res = d(w)
print(res)
print(res.sum(dim=-1))

tensor([[0.1546, 0.0545, 0.2186, 0.5723],
        [0.2369, 0.1055, 0.5400, 0.1176],
        [0.1910, 0.1514, 0.0653, 0.5923],
        [0.5418, 0.0481, 0.0398, 0.3704]])
tensor([1.0000, 1.0000, 1.0000, 1.0000])
tensor([[0.3092, 0.1090, 0.4372, 0.0000],
        [0.0000, 0.0000, 1.0800, 0.0000],
        [0.3819, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.7408]])
tensor([0.8554, 1.0800, 0.3819, 0.7408])


That's the part worth seeing. Row sums went from `[1, 1, 1, 1]` to `[0.38, 0.31, 1.84, 0.00]`.

So dropout on attention weights breaks the property softmax just established. Position 3 now over-weights its history by 1.84x; position 4 dropped _every_ connection, so its output is the zero vector this step — that token gathers nothing at all.

That's intended, not a bug. Each step, every token is forced to work with a random subset of the tokens it can see, so the model can't become dependent on any single attention edge. And since `eval()` is the identity, the weights sum to exactly 1 again whenever you actually measure or generate.

In [328]:
T = 4
tril = torch.tril(torch.ones(T, T))
w = torch.softmax(
    torch.randn(T, T).masked_fill(tril == 0, float("-inf")),
    dim=-1,
)
print(w)
print(w.sum(dim=-1))
res = d(w)
print(res)
print(res.sum(dim=-1))

tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.7113, 0.2887, 0.0000, 0.0000],
        [0.4614, 0.2845, 0.2540, 0.0000],
        [0.1128, 0.2475, 0.1902, 0.4495]])
tensor([1.0000, 1.0000, 1.0000, 1.0000])
tensor([[0.0000, 0.0000, 0.0000, 0.0000],
        [1.4226, 0.5774, 0.0000, 0.0000],
        [0.0000, 0.5691, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.8991]])
tensor([0.0000, 2.0000, 0.5691, 0.8991])


Right — row 0 has exactly one connection (itself, weight 1.0), so dropout is all-or-nothing there: `2.0` or `0`. No middle ground.

That's the asymmetry: early positions have few edges and get hit hard, later positions have many and lose only a fraction. Position 0 loses its entire context 50% of the time at `p=0.5`.

The reason that's survivable is the residual. `x = x + self.attn(self.ln1(x))` — when the attention output is zeroed, `x` still passes through untouched, so the token keeps its own identity and position. Dropout can delete what a token _gathered_, never what it _is_.

That's also why `p=0.2` rather than `0.5` in a transformer. With `block_size=256`, later tokens shrug it off, but early ones are fragile.

In [329]:
x = torch.arange(1.0, 11.0)
d = nn.Dropout(0.4)
d.train()
out = d(x)

print(x)
print(out)
print(out / x)  # the multiplier applied to each element
print(torch.stack([d(x) for _ in range(10000)]).mean(0))  # average over many draws

tensor([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.])
tensor([ 1.6667,  3.3333,  5.0000,  6.6667,  8.3333,  0.0000,  0.0000, 13.3333,
        15.0000, 16.6667])
tensor([1.6667, 1.6667, 1.6667, 1.6667, 1.6667, 0.0000, 0.0000, 1.6667, 1.6667,
        1.6667])
tensor([ 1.0052,  2.0110,  3.0075,  3.9533,  4.9450,  5.9520,  6.9078,  8.0413,
         8.9175, 10.0533])


Three things to look for:

*   `out / x` is only ever `0.0` or `1.6667` — dropout multiplies each element by one of exactly two values. `1.6667 = 1/(1-0.4)`.
*   The mask is per-element and independent, so the number of survivors varies run to run; `p` is a probability, not a quota.
*   The last line comes back ≈ `[1, 2, 3, ..., 10]` — your original `x`. Each element is `1.6667x` 60% of the time and `0` 40% of the time, and `0.6 × 1.6667 = 1.0`. That's what the scaling buys: the layer changes any individual forward pass, but not the expected value.

In [330]:
x = torch.arange(1.0, 11.0)
d = nn.Dropout(0.5)
d.train()
out = d(x)

print(x)
print(out)
print(out / x)  # the multiplier applied to each element
print(torch.stack([d(x) for _ in range(10000)]).mean(0))  # average over many draws

tensor([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.])
tensor([ 2.,  0.,  0.,  0.,  0., 12., 14., 16., 18.,  0.])
tensor([2., 0., 0., 0., 0., 2., 2., 2., 2., 0.])
tensor([0.9924, 2.0004, 3.0966, 3.9984, 4.9880, 6.0036, 7.0784, 7.9568, 8.8578,
        9.7840])


In [331]:
def my_dropout(x, p, training=True):
    if not training or p == 0.0:
        return x
    mask = (torch.rand_like(x) > p).float()
    return x * mask / (1 - p)
p = 0.5
x = torch.ones(4, 6)
res = my_dropout(x, p)
res


tensor([[0., 2., 2., 0., 0., 0.],
        [0., 2., 0., 0., 2., 2.],
        [0., 0., 0., 2., 0., 0.],
        [2., 2., 0., 0., 2., 2.]])

In [332]:
res = my_dropout(x, p, training=False)
res

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])

Two details worth noting in your implementation:

*   `training=True` as a parameter is what `nn.Module` handles for you via `self.training`, flipped by `.train()`/`.eval()`. That flag is the entire difference between the two modes.
*   `torch.rand_like(x) > p` gives `P(keep) = 1-p`, since `rand` is uniform on \[0,1). Getting that comparison backwards is the classic bug — it'd silently invert your dropout rate, and at `p=0.2` you'd be keeping 20% instead of 80%.

In [333]:
p = 0.3
x = torch.rand(2000, 200) + 1.0  # all values in [1, 2], so out/x is well behaved
ref = nn.Dropout(p)
ref.train()

for name, out in [("mine", my_dropout(x, p)), ("nn.Dropout", ref(x))]:
    r = out / x
    nz = r[r != 0]
    # zero_frac should be p = 0.3
    # scale should be 1/(1-p) = 1/(1-0.3) = 1/0.7 = 1.4286
    # out mean should be same as x mean after dropout (x mean = 0.5 + 1.0) 
    print(f"{name:12} zero_frac={(out == 0).float().mean():.4f}  "
          f"scale={nz.min():.4f}..{nz.max():.4f}  mean={out.mean():.4f}")

mine         zero_frac=0.2999  scale=1.4286..1.4286  mean=1.5005
nn.Dropout   zero_frac=0.3002  scale=1.4286..1.4286  mean=1.5002


In [334]:
row = x[0][:10]
print(row)
print(torch.stack([my_dropout(row, p) for _ in range(2000)]).mean(dim=0))
print(torch.stack([ref(row) for _ in range(2000)]).mean(dim=0))

tensor([1.6307, 1.4717, 1.4727, 1.5861, 1.7376, 1.4836, 1.5993, 1.1065, 1.4385,
        1.9268])
tensor([1.6027, 1.4749, 1.4727, 1.5703, 1.7389, 1.4730, 1.5787, 1.0994, 1.4210,
        1.9062])
tensor([1.6027, 1.4665, 1.4906, 1.5601, 1.7525, 1.4592, 1.6267, 1.1215, 1.3984,
        1.9117])
